# Practice #2. "Trend predictive models"

Measure forecast error, then forecast: naive, exponential smoothing,
Holt, Holt-Winters.

Fill in the cells tagged `graded`, keeping every name and signature exactly as
given — they are graded automatically.

**Contract.** Every `*_forecast(train, index, ...)` returns a `pd.Series` on
`index` — the timestamps you want predictions for. It may read `train` only; it
never sees the test values.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing

def find_data_dir():
    """The repo's data/ directory, wherever the kernel happens to start."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "data" / "airline-passengers.csv").exists():
            return folder / "data"
    raise FileNotFoundError("data/ not found - run this notebook inside the repo")


DATA_DIR = globals().get("DATA_DIR", find_data_dir())

## 0. Data reading and splitting

In [ ]:
def load_series(path, time_col, value_col):
    """Read a CSV into a float Series named "y" on a DatetimeIndex."""
    # TODO: as in Practice 1 — select the columns before dropping NaN.
    raise NotImplementedError


def train_test_split_ts(series, test_size):
    """Return (train, test); the LAST `test_size` points are the test set.

    Never shuffle a time series — a shuffled split lets the model see the future.
    """
    # TODO
    raise NotImplementedError

In [ ]:
series = load_series(DATA_DIR / "daily-min-temperatures.csv", "Date", "Temp")
train, test = train_test_split_ts(series, test_size=365)
print(f"train: {len(train)} points, {train.index.min():%Y-%m-%d} to {train.index.max():%Y-%m-%d}")
print(f"test : {len(test)} points, {test.index.min():%Y-%m-%d} to {test.index.max():%Y-%m-%d}")

In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(train, label="train")
plt.plot(test, label="test")
plt.ylabel("y")
plt.legend();

## 1. Forecasting performance measures

Implement each one from its definition, then check against scikit-learn.

Keep the names straight: **MSE is squared error, RMSE is its square root.**
Different units — you cannot compare one to the other.

In [ ]:
def forecast_error(actual, predicted):
    """actual - predicted, per point, as a numpy array."""
    # TODO
    raise NotImplementedError


def forecast_bias(actual, predicted):
    """Mean forecast error. Positive = the forecast runs low."""
    # TODO
    raise NotImplementedError


def mae(actual, predicted):
    """Mean absolute error."""
    # TODO
    raise NotImplementedError


def mse(actual, predicted):
    """Mean squared error. Not the root — squared units."""
    # TODO
    raise NotImplementedError


def rmse(actual, predicted):
    """Root mean squared error — same units as the data."""
    # TODO
    raise NotImplementedError

In [ ]:
actual = [0.5, 0.5, 0.5, 0.5]
predicted = [0.7, 0.5, 0.3, 0.2]

print("errors      ", forecast_error(actual, predicted))
print("bias (MFE)  ", forecast_bias(actual, predicted))
print("MAE         ", mae(actual, predicted))
print("MSE         ", mse(actual, predicted))
print("RMSE        ", rmse(actual, predicted))

assert np.isclose(mae(actual, predicted), mean_absolute_error(actual, predicted))
assert np.isclose(mse(actual, predicted), mean_squared_error(actual, predicted))
print("\nagrees with scikit-learn")

**Question.** The bias is positive and the MAE is non-zero. What does
each tell you that the other does not?

In [ ]:
def plot_forecast(train, test, forecast, title):
    """Plot the tail of the training data next to the test data and a forecast."""
    plt.figure(figsize=(20, 5))
    plt.plot(train.iloc[-3 * len(test):], label="train")
    plt.plot(test, label="test")
    plt.plot(forecast, label="forecast")
    plt.title(f"{title} — RMSE {rmse(test, forecast):.3f}")
    plt.legend()
    plt.show()

## 2. Naive approaches

### 2.1 Naive forecast

The last observed value, carried forward. Every model below has to beat it or it
is not worth its complexity.

In [ ]:
def naive_forecast(train, index):
    """Last observed value, repeated over `index`."""
    # TODO
    raise NotImplementedError


def moving_average_forecast(train, index, window):
    """Mean of the last `window` observations, repeated over `index`."""
    # TODO
    raise NotImplementedError


def linear_regression_forecast(train, index):
    """Fit a line on 0..len(train)-1 and extrapolate over `index`."""
    # TODO: the future steps continue the training ones: len(train), +1, ...
    raise NotImplementedError

In [ ]:
plot_forecast(train, test, naive_forecast(train, test.index), "Naive")

### 2.2 Moving average and 2.3 linear regression

In [ ]:
plot_forecast(train, test, moving_average_forecast(train, test.index, 30),
              "Trailing MA (30)")
plot_forecast(train, test, linear_regression_forecast(train, test.index),
              "Linear regression")

Find the window size with the lowest RMSE on the test set.

**Question.** That tuned a hyper-parameter *on the test set*. Why is the
resulting RMSE not your expected future error, and what should you have split
off instead?

In [ ]:
scores = {w: rmse(test, moving_average_forecast(train, test.index, w))
          for w in (7, 14, 30, 60, 90, 180, 365)}
for window, score in sorted(scores.items(), key=lambda kv: kv[1]):
    print(f"window {window:>4}: RMSE {score:.4f}")

## 3. Simple Exponential Smoothing

In [ ]:
def ses_forecast(train, index, alpha):
    """SES forecast — a flat line at the final level."""
    # TODO: SimpleExpSmoothing(initialization_method="known",
    # initial_level=train.iloc[0]).fit(smoothing_level=alpha, optimized=False)
    raise NotImplementedError


def holt_forecast(train, index, alpha, beta, trend="add"):
    """Holt forecast — level plus an extrapolated slope."""
    # TODO: ExponentialSmoothing(trend=trend, seasonal=None)
    raise NotImplementedError


def holt_winters_forecast(train, index, seasonal_periods, trend="add",
                          seasonal="add"):
    """Holt-Winters forecast; let statsmodels optimise the parameters.

    Raise ValueError on less than two full seasons of training data.
    """
    # TODO
    raise NotImplementedError

In [ ]:
plot_forecast(train, test, ses_forecast(train, test.index, 0.3), "SES (alpha=0.3)")

**Question.** The SES forecast is a horizontal line. Why, from the
recursion? And when is a flat forecast the *right* answer?

## 4. Holt's Linear Trend Model

In [ ]:
plot_forecast(train, test, holt_forecast(train, test.index, 0.3, 0.1),
              "Holt (alpha=0.3, beta=0.1)")

**Question.** Holt is the more powerful model — did it beat naive here?
What does the answer say about daily minimum temperatures?

## 5. Holt-Winters' model

The series has a yearly cycle. On daily data that means `seasonal_periods=365` —
365 seasonal parameters, estimated from 9 years. Slow and badly determined.
Resampling to monthly means captures the same cycle with 12.

Match the resolution of the data to the resolution of the pattern.

In [ ]:
monthly = series.resample("MS").mean()
m_train, m_test = train_test_split_ts(monthly, test_size=24)
print(f"monthly: {len(monthly)} points; train {len(m_train)}, test {len(m_test)}")

hw = holt_winters_forecast(m_train, m_test.index, seasonal_periods=12)
plot_forecast(m_train, m_test, hw, "Holt-Winters (monthly, m=12)")

In [ ]:
baseline = naive_forecast(m_train, m_test.index)
print(f"naive         RMSE {rmse(m_test, baseline):.4f}")
print(f"Holt-Winters  RMSE {rmse(m_test, hw):.4f}")

Try `trend` and `seasonal` as `"add"` and `"mul"`.

**Questions.** Which combination wins? And why does multiplicative seasonality
make little sense in degrees Celsius — what happens to it near zero?

In [ ]:
# your code here — free exploration, not graded